# 보고서의 결과를 CPU로 읽기

이 노트북은 공개 CSV 두 개에서 **기록된 집계값**을 읽고 표와 그래프를 다시 만듭니다. GPU, 데이터셋 사진, 모델 가중치, Google Drive 연결이 필요하지 않습니다.

같은 내부 test 30,809장에 대해 보고서에 기록된 RMSE는 v3 0.155691 m, v4 0.142334 m입니다. MAE와 P95도 감소했지만, 절대오차 1 m 초과 영상은 28장에서 31장으로 늘었습니다. 아래 감소율은 보고서에 반올림해 표시된 값에서 계산합니다.

**여기서 재현하는 것은 집계값의 읽기·단위 변환·그림입니다.** 원래 이미지별 예측에서 RMSE·MAE·P95를 다시 계산하거나, 모델을 학습·평가하는 노트북이 아닙니다.

[출처와 한계](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/results/README.md) · [연구보고서](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/reports/research-report.pdf)


## 1. 실행 준비

Colab에서는 런타임을 기본 CPU로 두고 위에서 아래로 실행합니다. 저장소를 내려받아 실행하면 로컬 `results/`를 우선 사용합니다. 저장소 밖에서는 GitHub의 **공개 CSV 두 개만** 내려받습니다. 비공개 클라우드 ID나 개인 파일이 필요하지 않습니다.

표 계산은 Python 표준 라이브러리만 사용합니다. 그래프용 `matplotlib`가 없는 환경에서만 해당 패키지를 설치합니다. 글꼴 다운로드 없이 실행되도록 이 노트북의 그래프 라벨은 영어로 표시합니다. 배포된 [한글 그래프](https://github.com/Pirite08/dcc-2026-water-level-estimation/tree/main/assets)는 로컬 Pretendard로 생성했습니다.


In [1]:
import csv
import importlib.util
import io
import subprocess
import sys
import tempfile
from decimal import Decimal
from pathlib import Path
from urllib.request import urlopen

if importlib.util.find_spec("matplotlib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "matplotlib"])

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("준비 완료: CPU에서 공개 집계값만 읽습니다.")


준비 완료: CPU에서 공개 집계값만 읽습니다.


## 2. 공개 CSV 읽기

`overall_metrics.csv`는 PDF 16쪽 표 12의 m 단위 값을, `error_threshold_counts.csv`는 PDF 21쪽의 절대오차 초과 건수를 담습니다. 출처 버전은 **2026년 9월 21일 연구보고서**입니다. 다운로드 경로는 `Pirite08/dcc-2026-water-level-estimation`의 `main/results/`로 고정했습니다.


In [2]:
CSV_NAMES = ("overall_metrics.csv", "error_threshold_counts.csv")
RAW_BASE = "https://raw.githubusercontent.com/Pirite08/dcc-2026-water-level-estimation/main/results/"

# 저장소 루트 또는 notebooks/ 안에서 실행하는 경우를 모두 지원합니다.
local_candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((candidate for candidate in local_candidates
                  if all((candidate / "results" / name).is_file() for name in CSV_NAMES)), None)

if repo_root is not None:
    csv_texts = {name: (repo_root / "results" / name).read_text(encoding="utf-8")
                 for name in CSV_NAMES}
    print("입력: 로컬 저장소의 results/ CSV 두 개")
else:
    csv_texts = {}
    for name in CSV_NAMES:
        with urlopen(RAW_BASE + name, timeout=30) as response:
            payload = response.read(65537)
        if len(payload) > 65536:
            raise ValueError("예상보다 큰 CSV입니다. 공개 파일을 확인하세요.")
        csv_texts[name] = payload.decode("utf-8-sig")
    print("입력: GitHub main/results/의 공개 CSV 두 개")

metrics = list(csv.DictReader(io.StringIO(csv_texts[CSV_NAMES[0]])))
counts = list(csv.DictReader(io.StringIO(csv_texts[CSV_NAMES[1]])))
print(f"읽은 행: 전체 지표 {len(metrics)}개, 초과 조건 {len(counts)}개")


입력: 로컬 저장소의 results/ CSV 두 개
읽은 행: 전체 지표 3개, 초과 조건 3개


## 3. 집계값의 범위 확인

단위, 표본 수, 중복되지 않은 지표 이름, 누적 초과 건수의 순서를 확인합니다. 이 검사는 **공개 CSV의 구조와 내부 일관성**을 확인하며 원예측의 정확성을 검증하지 않습니다.


In [3]:
SAMPLE_COUNT = 30809
assert [row["metric"] for row in metrics] == ["RMSE", "MAE", "absolute_error_P95"]
assert [Decimal(row["absolute_error_threshold_m"]) for row in counts] == [
    Decimal("0.2"), Decimal("0.5"), Decimal("1.0")
]
for row in metrics + counts:
    assert int(row["test_sample_count"]) == SAMPLE_COUNT
    assert row["source_id"] == "report_2026_09_21"
for row in metrics:
    assert row["unit"] == "m"
    for key in ("baseline_v3", "improved_v4"):
        value = Decimal(row[key])
        assert value.is_finite() and value >= 0
for row in counts:
    assert row["comparison"] == "strictly_greater_than"
    assert row["aggregation"] == "nested_cumulative"
    assert all(0 <= int(row[key]) <= SAMPLE_COUNT
               for key in ("baseline_v3_count", "improved_v4_count"))
for key in ("baseline_v3_count", "improved_v4_count"):
    values = [int(row[key]) for row in counts]
    assert values == sorted(values, reverse=True)
print("통과: m 단위, 내부 test 30,809장, 지표 3개, 누적 초과 조건 3개")
print("한계: 원예측 파일이 없어 지표 자체를 독립적으로 재계산한 검사는 아닙니다.")


통과: m 단위, 내부 test 30,809장, 지표 3개, 누적 초과 조건 3개
한계: 원예측 파일이 없어 지표 자체를 독립적으로 재계산한 검사는 아닙니다.


## 4. 기록된 수치와 변화량 읽기

감소율은 `(v3 − v4) / v3 × 100`입니다. m를 cm로 바꿀 때는 100을 곱합니다. CSV의 m 값은 보고서의 소수점 여섯 자리 표기를 유지하고, 아래 cm 값은 그 수치를 변환한 값입니다.


In [4]:
for row in metrics:
    before, after = Decimal(row["baseline_v3"]), Decimal(row["improved_v4"])
    decrease = (before - after) / before * 100
    print(f'{row["metric"]}: {before:.6f} → {after:.6f} m')
    print(f'  {before * 100:.4f} → {after * 100:.4f} cm | 감소율 {decrease:.2f}%')
print("감소율의 입력은 보고서에 표시된 반올림 집계값입니다.")


RMSE: 0.155691 → 0.142334 m
  15.5691 → 14.2334 cm | 감소율 8.58%
MAE: 0.056090 → 0.054499 m
  5.6090 → 5.4499 cm | 감소율 2.84%
absolute_error_P95: 0.154746 → 0.126563 m
  15.4746 → 12.6563 cm | 감소율 18.21%
감소율의 입력은 보고서에 표시된 반올림 집계값입니다.


전체 지표만으로는 큰 실패의 변화를 알기 어렵습니다. 아래 조건은 모두 **초과(>)**이며 서로 겹칩니다. 1 m 초과 영상은 0.5 m 초과와 0.2 m 초과에도 포함되므로 건수를 합산하지 않습니다.


In [5]:
for row in counts:
    before, after = int(row["baseline_v3_count"]), int(row["improved_v4_count"])
    print(f'{row["absolute_error_threshold_m"]} m 초과: {before:,} → {after:,}장 | 변화 {after-before:+,}장')
print("주의: 세 행은 서로 겹치는 누적 조건입니다. 합계를 만들지 않습니다.")


0.2 m 초과: 966 → 525장 | 변화 -441장
0.5 m 초과: 216 → 178장 | 변화 -38장
1.0 m 초과: 28 → 31장 | 변화 +3장
주의: 세 행은 서로 겹치는 누적 조건입니다. 합계를 만들지 않습니다.


## 5. CPU에서 그래프 다시 만들기

공개 CSV로 PNG와 SVG 두 쌍을 생성합니다. 전체 지표는 세 패널 모두 0~18 cm, 초과 건수는 패널마다 다른 0 시작 축을 사용합니다. 원예측이나 사진을 추가로 읽지 않습니다.

생성 위치는 실행 환경의 임시 폴더 `OUTPUT_DIR`입니다. 이미지 페이로드를 공개 노트북 안에 넣지 않고 파일로 저장하므로 저장된 노트북에는 수치 출력만 남습니다. Colab에서 파일을 내려받으려면 실행 후 `from google.colab import files; files.download(str(OUTPUT_DIR / "recorded_metrics.png"))`를 별도 셀에서 실행할 수 있습니다.


In [6]:
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="dcc-recorded-results-"))
MINT, BASELINE, INK, MUTED, GRID = "#80AA90", "#CCD3CE", "#17221C", "#56625B", "#E4E9E5"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 12,
                     "text.color": INK, "axes.labelcolor": MUTED,
                     "xtick.color": INK, "ytick.color": MUTED,
                     "figure.facecolor": "#FCFDFC", "axes.facecolor": "#FCFDFC",
                     "savefig.facecolor": "#FCFDFC", "svg.fonttype": "path"})

def make_figure(rows, kind):
    figure = plt.figure(figsize=(12, 6.4))
    is_metric = kind == "metrics"
    title = ("All three recorded aggregate errors decreased" if is_metric
             else "Errors over 1 m increased from 28 to 31 images")
    subtitle = ("Same internal test: 30,809 images | Lower is better | All axes: 0-18 cm" if is_metric
                else "Same 30,809 images | Overlapping thresholds | Different y-axis ranges")
    figure.text(0.055, 0.925, title, fontsize=22, weight="bold")
    figure.text(0.055, 0.866, subtitle, fontsize=12, color=MUTED)
    labels = ["RMSE", "MAE", "Absolute error P95"]
    for index, row in enumerate(rows):
        left, center = 0.08 + index * 0.315, 0.198 + index * 0.315
        if is_metric:
            before, after = Decimal(row["baseline_v3"]), Decimal(row["improved_v4"])
            values = [float(before * 100), float(after * 100)]
            label, top, ticks, unit = labels[index], 18, [0, 5, 10, 15], "cm"
            change = f"{(before-after)/before*100:.2f}% decrease"
        else:
            values = [int(row["baseline_v3_count"]), int(row["improved_v4_count"])]
            label = f'Over {Decimal(row["absolute_error_threshold_m"]):g} m'
            top, ticks = [(1100, [0, 500, 1000]), (250, [0, 100, 200]),
                          (40, [0, 10, 20, 30, 40])][index]
            unit, change = "images", f"{values[1]-values[0]:+,} images"
        figure.text(center, 0.75, label, ha="center", fontsize=17, weight="bold")
        axis = figure.add_axes([left, 0.285, 0.245, 0.365])
        axis.set_ylim(0, top)
        axis.set_xlim(-0.65, 1.65)
        axis.set_yticks(ticks)
        axis.set_ylabel(unit)
        axis.set_xticks([0, 1], ["Initial model\n(v3)", "Revised model\n(v4)"], fontsize=11)
        axis.tick_params(axis="both", length=0, pad=9)
        axis.grid(axis="y", color=GRID, linewidth=0.8)
        axis.set_axisbelow(True)
        for side in ("top", "right", "left"):
            axis.spines[side].set_visible(False)
        axis.spines["bottom"].set_color("#B9C4BC")
        axis.bar([0, 1], values, width=0.52, color=[BASELINE, MINT], zorder=3)
        for position, value in enumerate(values):
            axis.text(position, value + top * 0.037,
                      f"{value:.2f}" if is_metric else f"{value:,}",
                      ha="center", fontsize=15, weight="bold")
        figure.text(center, 0.14, change, ha="center", fontsize=14, weight="bold")
    source = ("Source: research report dated 2026-09-21, p.16, Table 12 | m converted to cm"
              if is_metric else "Source: research report dated 2026-09-21, p.21 | Every y-axis starts at zero")
    limit = ("Changes use rounded report aggregates, not recomputed image-level predictions."
             if is_metric else "Overlapping cumulative counts must not be summed. Predictions were not reevaluated.")
    figure.text(0.055, 0.048, source + chr(10) + limit, fontsize=10.5, color=MUTED, linespacing=1.55)
    stem = "recorded_metrics" if is_metric else "recorded_error_counts"
    for extension in ("png", "svg"):
        figure.savefig(OUTPUT_DIR / f"{stem}.{extension}", dpi=180)
    plt.close(figure)
    print(f"생성: {stem}.png, {stem}.svg")

make_figure(metrics, "metrics")
make_figure(counts, "counts")
assert all((OUTPUT_DIR / f"{stem}.{extension}").stat().st_size > 0
           for stem in ("recorded_metrics", "recorded_error_counts")
           for extension in ("png", "svg"))
print("그림 파일 4개 생성 완료. 표와 그래프는 같은 공개 CSV에서 만들어졌습니다.")


생성: recorded_metrics.png, recorded_metrics.svg
생성: recorded_error_counts.png, recorded_error_counts.svg
그림 파일 4개 생성 완료. 표와 그래프는 같은 공개 CSV에서 만들어졌습니다.


## 6. 해석할 때 남겨 둘 한계

- RMSE, MAE, P95가 감소했다는 것은 **이 내부 test의 보고서 기록**입니다. 원예측 파일을 복구하지 못했으므로 이 노트북에서 지표 자체를 독립적으로 재계산하지 못합니다.
- 초과 건수는 0.2 m에서 966→525, 0.5 m에서 216→178로 줄었지만 1 m에서는 28→31로 늘었습니다. 평균적인 지표 감소와 큰 실패의 잔존을 함께 읽어야 합니다.
- 이 test는 v3 오류 분석에 사용된 내부 고정 평가 자료입니다. 새로운 장소나 촬영 환경의 일반화 성능을 검증한 결과로 읽으면 안 됩니다.
- 공개 준비 과정에서 모델 재학습이나 추론을 실행하지 않았습니다. 원본 실험 실행 기록과 이 집계값 읽기 실행은 구분합니다.

한글 배포 그림: [전체 지표](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/assets/recorded_metrics.png) · [초과 건수](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/assets/recorded_error_counts.png)
